# Bikeztagram — MiniMax Music 3 £0 Kaggle T4x2 Forge Test

Replacement for the unreliable Colab test. Kaggle T4x2 provides two 16 GB GPUs, matching MiniMax Music 3's documented dual-GPU topology. Generation runs locally on those GPUs; there is no hosted music-generation quota or ZeroGPU inference.

**Use Settings → Accelerator → T4 x2. No payment details.**


In [ ]:
import os, subprocess, shutil, urllib.request
print('=== BIKEZTAGRAM GPU PREFLIGHT ===')
gpus=subprocess.check_output(['nvidia-smi','--query-gpu=index,name,memory.total','--format=csv,noheader'],text=True).strip()
print(gpus)
gpu_count=len([x for x in gpus.splitlines() if x.strip()])
if gpu_count < 2: raise RuntimeError(f'Need NVIDIA T4 x2; detected {gpu_count} GPU(s). Go to Settings → Accelerator → GPU T4 x2.')
for p in ['/kaggle/working','/kaggle/temp']:
 u=shutil.disk_usage(p); print(p,'free GB:',round(u.free/1024**3,2))
try:
 urllib.request.urlopen('https://huggingface.co',timeout=10)
 print('Internet: OK')
except Exception as e: raise RuntimeError('Kaggle Internet is OFF. Use Settings → Turn on internet, then rerun this cell.') from e


In [ ]:
import subprocess,sys,shutil,os
print('=== INSTALL SGLANG-OMNI ===')
# Kaggle can install uv but not expose its executable on PATH. Invoke it through Python instead.
subprocess.run([sys.executable,'-m','pip','install','-q','--no-input','-U','pip','uv'],check=True)
subprocess.run([sys.executable,'-m','uv','pip','install','--system','--prerelease=allow','sglang-omni==0.1.7'],check=True)
sgl=shutil.which('sgl-omni')
if not sgl: raise RuntimeError('sgl-omni was installed but is not on PATH.')
print('SGLang-Omni installed:',sgl)


In [ ]:
import os,subprocess,shutil
os.environ['HF_HOME']='/kaggle/temp/huggingface'; os.environ['HF_HUB_CACHE']='/kaggle/temp/huggingface/hub'
os.makedirs('/kaggle/temp/huggingface',exist_ok=True)
old_pid='/kaggle/temp/music3_server.pid'
if os.path.exists(old_pid):
 try:
  pid=int(open(old_pid).read().strip()); os.kill(pid,0); print('Existing Music 3 server is still running, PID',pid)
 except Exception: pass
else:
 log=open('/kaggle/temp/music3_server.log','a')
 sgl=shutil.which('sgl-omni') or 'sgl-omni'
 p=subprocess.Popen(['bash','-lc',f'CUDA_VISIBLE_DEVICES=0,1 {sgl} serve --model-path MiniMaxAI/MiniMax-Music3 --port 8000'],stdout=log,stderr=subprocess.STDOUT,start_new_session=True,env=os.environ.copy())
 open(old_pid,'w').write(str(p.pid))
 print('MiniMax Music 3 server started, PID',p.pid)
print('Model cache:',os.environ['HF_HOME'])


In [ ]:
# LIVE STATUS — rerun safely
import subprocess,shutil
from pathlib import Path
for p in ['/kaggle/temp','/kaggle/working']:
 u=shutil.disk_usage(p); print(p,'free GB:',round(u.free/1024**3,2))
print('\nGPU:'); print(subprocess.check_output(['nvidia-smi','--query-gpu=index,name,utilization.gpu,memory.used,memory.total','--format=csv,noheader'],text=True))
lp=Path('/kaggle/temp/music3_server.log'); print('\nLOG:\n'+(lp.read_text()[-7000:] if lp.exists() else 'No log'))
r=subprocess.run(['curl','-s','-m','5','http://127.0.0.1:8000/v1/models'],capture_output=True,text=True)
print('\nSERVER:', 'READY' if r.returncode==0 and r.stdout else 'STARTING')


In [ ]:
# WAIT + REAL 30-SECOND RENDER
import time,subprocess,requests
ready=False
for i in range(180):
 r=subprocess.run(['curl','-s','-m','5','http://127.0.0.1:8000/v1/models'],capture_output=True,text=True)
 if r.returncode==0 and r.stdout: ready=True; print('SERVER READY'); break
 if i%6==0: print('still starting —',i*5,'seconds')
 time.sleep(5)
if not ready: raise RuntimeError('Server not ready; run LIVE STATUS and inspect the log.')
lyrics='''[Verse]\nNeon cuts across the road\nEngine singing through the night\nCity fading in the mirrors\nChasing every line of light\n\n[Chorus]\nRide into the open dark\nChase the road and chase the light\nNo turning back, the engine calls\nWe own the road tonight'''
instructions='Cinematic electronic rock motorcycle soundtrack, 105 BPM, D minor, deep punchy drums, pulsing bass, distorted electric guitar, atmospheric synths, dramatic build and a huge energetic chorus, polished modern production.'
payload={'model':'MiniMaxAI/MiniMax-Music3','input':lyrics,'instructions':instructions,'seed':1001,'max_new_tokens':750}
resp=requests.post('http://127.0.0.1:8000/v1/audio/speech',json=payload,timeout=1800); resp.raise_for_status()
out='/kaggle/working/bikeztagram_minimax_music3_test.wav'; open(out,'wb').write(resp.content)
print('DONE:',out,'bytes:',len(resp.content))


## Result

The final cell writes the first real MiniMax Music 3 WAV to `/kaggle/working`. This is the proof-of-life render; the next step is wiring the same runtime behind Bikeztagram's private Music Engine with wake/render/shutdown.
